# BıRAG `v0.1.1` — Gemma 4 E4B LoRA ince ayarı (Unsloth, Colab)

Unsloth'un [Gemma4 (E4B) Text](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Gemma4_(E4B)-Text.ipynb) defterinin akışı: **kur → modeli yükle → veri → LoRA → eğit → üret → kaydet**.

**Bir koşu = bir kol × bir tohum.** 2. hücrede `KOL` ve `TOHUM`'u seçin, *Çalışma zamanı → Tümünü çalıştır*.
Faz 4'ün tamamı 16 koşu; bu sırayla (iki kol aynı oturumda, dönüşümlü):

| | t7 | t13 | t23 | t31 | t37 | t41 | t43 | t47 |
|---|---|---|---|---|---|---|---|---|
| `v010u` (`datasets/v0.0.22` = `v0.1.0`) | 1 | 3 | 5 | 7 | 9 | 11 | 13 | 15 |
| `v011u` (`datasets/v0.1.1`) | 2 | 4 | 6 | 8 | 10 | 12 | 14 | 16 |

⚠️ **GPU: A100 ya da L4** (bf16, 4-bit yok). Her şey Drive'a yazılır; oturum koparsa aynı `KOL`/`TOHUM` ile yeniden çalıştırın — biten adım atlanır, yarım eğitim son kontrol noktasından sürer.
Rehber: `docs/colab-egitim-rehberi.md` · ön kayıt: EK-3 (`configs/deney/2026-09-30-v011-on-kayit-ek3.json`) — bu defter mühürlü, tarifi değiştirmeyin.

### 1. Drive ve kurulum
**İlk koşu** en son Unsloth'u kurar; kurulan sürümler 8. hücrede `ortam-referans.json`'a yazılır. **Sonraki koşular** aynı sürümleri kurar (EK-3) — iki kol, oturumlar arasında Unsloth güncellense de aynı kütüphanelerle eğitilir. `torch` Colab'ın kendi sürümüdür; değişirse 8. hücre durur.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json, os
REF = "/content/drive/MyDrive/birag-v011/ortam-referans.json"      # 2. hücredeki DRIVE ile aynı yer
if os.path.exists(REF):
    PIN = " ".join(f"{p}=={v}" for p, v in json.load(open(REF))["surumler"].items() if v and p != "torch")
    print("referans sürümleri kuruluyor:", PIN)
    !pip install -q {PIN}
else:
    print("ilk koşu: en son unsloth kuruluyor")
    !pip install -q unsloth

### 2. Parametreler
Tarif **değiştirilmez** — ön kayıttaki `e3` tarifidir (`configs/training/*-t{tohum}.yaml`). Değişen yalnız `KOL` ve `TOHUM`.

In [ ]:
KOL   = "v010u"        # "v010u" | "v011u"
TOHUM = 7              # 7 · 13 · 23 · 31 · 37 · 41 · 43 · 47

MODEL_ID = "unsloth/gemma-4-E4B-it"          # 16 koşuda AYNI olmalı
DRIVE    = "/content/drive/MyDrive/birag-v011"
REPO     = "/content/birag-data-generation"

VERI = {"v010u": "datasets/v0.0.22/train.jsonl", "v011u": "datasets/v0.1.1/train.jsonl"}[KOL]

# ön kayıtlı tarif (e3) — MLX'ten aktarım: lora_alpha = scale 20 × r 8
R, LORA_ALPHA, UST_KATMAN = 8, 160, 8        # üstten 8 dil katmanı, yalnız q_proj + o_proj
ADIM, LR, MAX_SEQ = 2538, 1e-5, 2048         # 846 kayıt × 3 tur, batch 1, sabit LR
URETIM = {"thinking": False, "max_tokens": 1024, "do_sample": False}   # açgözlü — kayıtlı koşularla aynı

### 3. Depo

In [ ]:
import sys, subprocess
if not os.path.exists(REPO):
    !git clone -q https://github.com/enveryildirim/birag-data-generation.git {REPO}
else:
    !git -C {REPO} pull -q --ff-only
GIT_REV = subprocess.run(["git", "-C", REPO, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
sys.path.insert(0, f"{REPO}/src")
print("depo:", GIT_REV)

from pathlib import Path
KOSU = Path(DRIVE) / "runs" / f"{KOL}-t{TOHUM}"
ADAPTER = KOSU / "adapter"
KOSU.mkdir(parents=True, exist_ok=True)

### 4. Modeli yükle
**bf16, 4-bit yok** (Unsloth örneğindeki `load_in_4bit = True` burada `False`). Model revizyonu ilk koşuda sabitlenir ve `ortam-referans.json`'a yazılır; sonraki koşular oradan okur.

In [ ]:
from unsloth import FastModel
import torch

REFERANS = Path(DRIVE) / "ortam-referans.json"
if REFERANS.exists():
    REV = json.loads(REFERANS.read_text())["model_revizyon"]
else:
    from huggingface_hub import model_info
    REV = model_info(MODEL_ID).sha

def taban_yukle():
    return FastModel.from_pretrained(model_name=MODEL_ID, revision=REV, max_seq_length=MAX_SEQ,
                                     dtype=torch.bfloat16, load_in_4bit=False, full_finetuning=False)

gpu = torch.cuda.get_device_properties(0)
assert gpu.total_memory / 1e9 >= 22, f"GPU {gpu.name} yetmez — A100 ya da L4 seçin"
print("GPU:", gpu.name, f"{gpu.total_memory/1e9:.0f} GB · model:", MODEL_ID, REV[:12])

### 5. Sohbet şablonu
Unsloth örneği `get_chat_template(tokenizer, "gemma-4")` kullanıyor. Biz **kendi eğitim şablonumuzu** (`configs/chat_template_train.jinja`, K44) kullanıyoruz: düşünme son asistan turuna `<|channel>thought … <channel|>` olarak gömülü ve kayıtlı `e3` koşuları bu şablonla eğitildi.
Unsloth'un uyarısı da aynı: *eğitimde hangi şablon kullanıldıysa çıkarımda da o.*

In [ ]:
SABLON = Path(REPO, "configs/chat_template_train.jinja")

def sablonlu(tokenizer):
    tok = getattr(tokenizer, "tokenizer", tokenizer)       # Gemma 4 işlemci döndürür → metin tokenizer'ı
    tok.chat_template = SABLON.read_text(encoding="utf-8")
    return tok

import colab_egitim as ce                                   # yalnız şablon denetimi için (35 referans dizge)

### 6. Veri
Bölme `src/train.py`'nin kendisiyle yapılır (`random.Random(7)`, %20 doğrulama ⇒ 846 / 212) — kayıtlı koşularla aynı kayıtlar.

**Kayıp yalnız SON asistan turunda.** Unsloth'un `train_on_responses_only`'si bütün asistan turlarını eğitir; MLX'in `mask_prompt`'u yalnız sonuncuyu. 405 çok turlu kayıtta bu fark eder ⇒ maskeyi elle kuruyoruz.

In [ ]:
import tempfile
from train import prepare_data

with tempfile.TemporaryDirectory() as d:
    n_tr, n_va = prepare_data(Path(REPO, VERI), Path(d), 0.2, 7)
    oku = lambda ad: [json.loads(x)["messages"] for x in open(f"{d}/{ad}.jsonl") if x.strip()]
    egitim_msj, dogrulama_msj = oku("train"), oku("valid")
print(f"{VERI}: {n_tr} eğitim / {n_va} doğrulama")

def kodla(tok, msgs):
    tam = tok(tok.apply_chat_template(msgs, tokenize=False), add_special_tokens=False)["input_ids"]
    on  = tok(tok.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True),
              add_special_tokens=False)["input_ids"]
    assert tam[:len(on)] == on, "önek hizası bozuk"
    assert len(tam) <= MAX_SEQ, f"{len(tam)} jeton > {MAX_SEQ} — sessiz kırpma olmaz, durun"
    return {"input_ids": tam, "attention_mask": [1] * len(tam), "labels": [-100] * len(on) + tam[len(on):]}

### 7. Eğit
LoRA: **üstten 8 dil katmanının `q_proj` + `o_proj`'u**, görü ve ses kuleleri hariç. Hedefler tam adla verilir — `["q_proj", "o_proj"]` gibi kısa adlar PEFT'te sonek eşlemesiyle görü kulesine de gider.
AdamW (β 0,9/0,999 · wd 0,01), sabit LR, ısınma yok, gradyan kırpma yok. Adapter zaten varsa bu hücre atlar.

In [ ]:
import re, time, transformers
from datasets import Dataset

if (ADAPTER / "adapter_config.json").exists():
    print(f"✅ {KOL}-t{TOHUM} zaten eğitilmiş — 8. hücreye geçin")
else:
    transformers.set_seed(TOHUM)
    model, tokenizer = taban_yukle()
    print("şablon denetimi:", ce.sablon_referansi_denetle(tokenizer), "dizge referansla aynı")
    tok = sablonlu(tokenizer)

    # LoRA hedefleri: dil katmanları, üstten 8, q/o
    katman = {}
    for ad, _ in model.named_modules():
        m = re.search(r"\.layers\.(\d+)\.self_attn\.(q_proj|o_proj)$", ad)
        if m and not re.search(r"vision|audio", ad):
            katman.setdefault(int(m.group(1)), []).append(ad)
    ust = sorted(katman)[-UST_KATMAN:]
    HEDEF = [ad for k in ust for ad in katman[k]]
    print(f"dil katmanı {len(katman)} · LoRA katmanları {ust[0]}–{ust[-1]} · {len(HEDEF)} modül")

    model = FastModel.get_peft_model(model, r=R, lora_alpha=LORA_ALPHA, lora_dropout=0.0, bias="none",
                                     target_modules=HEDEF, use_gradient_checkpointing="unsloth", random_state=TOHUM)
    lora = [ad for ad, mod in model.named_modules() if hasattr(mod, "lora_A") and len(mod.lora_A)]
    assert len(lora) == len(HEDEF) and not any(re.search("vision|audio", a) for a in lora), f"LoRA yanlış yerde: {lora[:3]}"
    model.print_trainable_parameters()

    dtr = Dataset.from_list([kodla(tok, m) for m in egitim_msj])
    dva = Dataset.from_list([kodla(tok, m) for m in dogrulama_msj])

    trainer = transformers.Trainer(
        model=model, train_dataset=dtr, eval_dataset=dva,
        data_collator=transformers.DataCollatorForSeq2Seq(tok, padding=True),
        args=transformers.TrainingArguments(
            output_dir=str(KOSU / "ckpt"), per_device_train_batch_size=1, per_device_eval_batch_size=1,
            max_steps=ADIM, learning_rate=LR, lr_scheduler_type="constant", warmup_steps=0,
            optim="adamw_torch", weight_decay=0.01, adam_beta1=0.9, adam_beta2=0.999, adam_epsilon=1e-8,
            max_grad_norm=0.0, bf16=True, logging_steps=50, eval_strategy="steps", eval_steps=423,
            save_strategy="steps", save_steps=846, seed=TOHUM, data_seed=TOHUM,
            report_to="none", remove_unused_columns=False))

    t0 = time.time()
    trainer.train(resume_from_checkpoint=any((KOSU / "ckpt").glob("checkpoint-*")))
    model.save_pretrained(str(ADAPTER))
    dk = round((time.time() - t0) / 60, 1)
    (KOSU / "egitim.json").write_text(json.dumps({"kol": KOL, "tohum": TOHUM, "veri": VERI, "hedefler": HEDEF,
        "egitim_kayit": len(dtr), "dogrulama_kayit": len(dva), "sure_dk": dk, "git": GIT_REV,
        "log": trainer.state.log_history}, ensure_ascii=False, indent=1, default=str))
    son = [x["eval_loss"] for x in trainer.state.log_history if "eval_loss" in x]
    print(f"✅ {KOL}-t{TOHUM}: {dk} dk · son eval_loss {son[-1] if son else '—'}")
    del trainer, model; torch.cuda.empty_cache()

### 8. Ortam kaydı
İki kolun **aynı ortamda** eğitilip üretildiğinin kanıtı. İlk koşu referansı yazar; sonraki koşularda model, revizyon, şablon, kütüphane sürümleri ya da üretim ayarı değiştiyse **durur** (çözümleme de aynı denetimi yapar).

In [ ]:
import hashlib, importlib.metadata as md
def surum(p):
    try: return md.version(p)
    except md.PackageNotFoundError: return None

ORTAM = {"model_id": MODEL_ID, "model_revizyon": REV, "lora_yolu": "unsloth", "git_rev": GIT_REV,
         "kosucu": "notebooks/v011-basit-egitim.ipynb",
         "sablon_sha256_16": hashlib.sha256(SABLON.read_bytes()).hexdigest()[:16],
         "surumler": {p: surum(p) for p in ce.KUTUPHANE}, "uretim": URETIM,
         "gpu": torch.cuda.get_device_name(0), "cuda": torch.version.cuda, "python": sys.version.split()[0]}
print("ortam:", ce.ortam_denetle(ORTAM, REFERANS))

### 9. Üret — 6 eksen + çok turlu
Taban + adapter yeniden yüklenir (eğitim atlanmış olsa da aynı yol). **Açgözlü, `thinking` kapalı, 1024 jeton** — Gemma'nın önerdiği `temperature = 1.0, top_p = 0.95, top_k = 64` burada kullanılmaz: ölçüm tekrarlanabilir olmalı ve kayıtlı koşularla aynı ayarda.
Çok turlu konuşmada geçmişe yalnız görünen cevap konur, düşünme konmaz (Gemma 4 kuralı). **Puanlama burada yapılmaz** — veri makinesinde.

In [ ]:
from datetime import datetime
from peft import PeftModel
from golden_eval import prompt_kur, cikti_ayir

model, tokenizer = taban_yukle()
ce.sablon_referansi_denetle(tokenizer)
tok = sablonlu(tokenizer)
model = PeftModel.from_pretrained(model, str(ADAPTER)); model.eval()
eos = model.generation_config.eos_token_id
eos = sorted(set(eos if isinstance(eos, (list, tuple)) else [eos]))

def uret(oge):
    istem = prompt_kur(oge, tok, URETIM["thinking"])
    ids = tok(istem, add_special_tokens=False)["input_ids"]
    x = torch.tensor([ids], device=model.device); t1 = time.time()
    with torch.no_grad():
        y = model.generate(input_ids=x, attention_mask=torch.ones_like(x), max_new_tokens=URETIM["max_tokens"],
                           do_sample=False, eos_token_id=eos, pad_token_id=tok.pad_token_id or eos[0])
    yeni = y[0, len(ids):].tolist(); durdu = bool(yeni) and yeni[-1] in eos
    while yeni and yeni[-1] in eos: yeni.pop()
    ham = tok.decode(yeni, skip_special_tokens=False)
    dus, cevap, kapandi = cikti_ayir(ham)
    return {"id": oge["id"], "ham": ham, "thinking": dus, "cevap": cevap, "thinking_kapandi": kapandi,
            "sure_sn": round(time.time() - t1, 1), "uretim_token": len(yeni) + int(durdu), "kesildi": not durdu}

def yaz(alt, kisa, set_yol, satirlar, **ek):
    d = Path(DRIVE) / alt / f"{datetime.now():%Y%m%d-%H%M%S}-{KOL}-t{TOHUM}-{kisa}"; d.mkdir(parents=True)
    satir = "".join(json.dumps(r, ensure_ascii=False) + "\n" for r in satirlar)
    (d / "sonuclar.jsonl").write_text(satir); (d / "ham.jsonl").write_text(satir)
    (d / "kosu.json").write_text(json.dumps({"etiket": f"{KOL}-t{TOHUM}-{kisa}", "set": set_yol,
        "set_sha256_16": hashlib.sha256(Path(REPO, set_yol).read_bytes()).hexdigest()[:16],
        "adapter": str(ADAPTER), "model": MODEL_ID, "thinking": URETIM["thinking"],
        "max_tokens": URETIM["max_tokens"], "tarih": datetime.now().isoformat(timespec="seconds"), **ek}, ensure_ascii=False, indent=1))
    (d / "ortam.json").write_text(json.dumps(ORTAM, ensure_ascii=False, indent=1))
    print("  ✅", d.name)

oku = lambda yol: [json.loads(x) for x in open(Path(REPO, yol), encoding="utf-8") if x.strip()]

for kisa, yol in ce.EKSEN:                     # safety · forget · context_fidelity · sycophancy · cfreal · cfo
    if any((Path(DRIVE) / "eksen-kosu").glob(f"*-{KOL}-t{TOHUM}-{kisa}/sonuclar.jsonl")):
        print(f"  {kisa}: zaten var"); continue
    t0 = time.time(); satirlar = [uret(o) for o in oku(yol)]
    yaz("eksen-kosu", kisa, yol, satirlar, oge=len(satirlar), uretim_sn=round(time.time() - t0))

if not any((Path(DRIVE) / "cok-turlu-kosu").glob(f"*-{KOL}-t{TOHUM}-cokturlu/sonuclar.jsonl")):
    t0 = time.time(); turlar = []
    for o in oku(ce.COK_TURLU):
        msgs = [dict(m) for m in o["messages"]]
        for k, u in enumerate(o["kullanici_turlari"], 1):
            msgs.append({"role": "user", "content": u})
            c = uret({"id": f"{o['id']}-t{k}", "messages": list(msgs)})
            turlar.append({"konusma": o["id"], "tur": k, "kullanici": u, **c})
            msgs.append({"role": "assistant", "content": c["cevap"]})      # geçmişe yalnız cevap
    yaz("cok-turlu-kosu", "cokturlu", ce.COK_TURLU, turlar, konusma=len(turlar) // 4, tur=len(turlar),
        uretim_sn=round(time.time() - t0))
else:
    print("  cokturlu: zaten var")

### 10. Durum ve paket
16 koşu bitince tek zip. Adapter'lar pakete girmez (hüküm için gerekmez), Drive'da kalır.

In [ ]:
import zipfile
bitti = [(k, t) for t in (7, 13, 23, 31, 37, 41, 43, 47) for k in ("v010u", "v011u")
         if len(list((Path(DRIVE) / "eksen-kosu").glob(f"*-{k}-t{t}-*"))) == 6
         and any((Path(DRIVE) / "cok-turlu-kosu").glob(f"*-{k}-t{t}-cokturlu"))]
print(f"üretimi tam koşu: {len(bitti)}/16")
if len(bitti) == 16:
    paket = Path(DRIVE) / "v011-colab-ciktilari.zip"
    with zipfile.ZipFile(paket, "w", zipfile.ZIP_DEFLATED) as z:
        for alt in ("eksen-kosu", "cok-turlu-kosu"):
            for f in (Path(DRIVE) / alt).rglob("*"):
                if f.is_file(): z.write(f, f.relative_to(DRIVE))
        for f in (Path(DRIVE) / "runs").glob("*/egitim.json"): z.write(f, f.relative_to(DRIVE))
        z.write(REFERANS, REFERANS.relative_to(DRIVE))
    print("✅ paket:", paket)